# **House Price Prediction Project**

## **Data Pre-processing**

### **Load Data and Libraries**

#### **Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### **Load Data**

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
df = pd.concat([train, test], ignore_index=True)
df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500.0
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500.0
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500.0
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000.0
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2006,WD,Normal,NaN
2915,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml,NaN
2916,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml,NaN
2917,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,NaN


### **Data Cleaning and Transformation** 

#### **Data Cleaning**

In [4]:
# 1. KATEGORIKAL ORDINAL
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_cols:
    if col != 'KitchenQual':
        df[col] = df[col].fillna('None')

df['KitchenQual'] = df['KitchenQual'].fillna(train['KitchenQual'].mode()[0])

qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
for col in qual_cols:
    df[col] = df[col].map(qual_mapping)

df['Functional'] = df['Functional'].fillna('Typ').map({'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8})
df['LandSlope'] = df['LandSlope'].fillna('Gtl').map({'Sev': 1, 'Mod': 2, 'Gtl': 3})

In [5]:
# 2. KATEGORIKAL NOMINAL
nominal_cols = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 
    'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 
    'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 
    'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 
    'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition'
]

none_fill_cols = ['Alley', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish', 'Fence', 'MiscFeature', 'MasVnrType']

for col in nominal_cols:
    if col in none_fill_cols:
        df[col] = df[col].fillna('None')
    else:
        df[col] = df[col].fillna(train[col].mode()[0])

for col in nominal_cols:
    df[col] = df[col].astype('category')

In [6]:
# 3. NUMERIK
num_cols_to_zero = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
for col in num_cols_to_zero:
    df[col] = df[col].fillna(0)

df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

frontage_median = train.groupby('Neighborhood')['LotFrontage'].median()
df['LotFrontage'] = df['LotFrontage'].fillna(df['Neighborhood'].map(frontage_median))
df['LotFrontage'] = df['LotFrontage'].fillna(train['LotFrontage'].median())

#### **Data Transformation**

In [7]:
# 4. TARGET TRANSFORMATION
df['SalePrice'] = np.log1p(df['SalePrice'])

print("Preprocessing selesai. Data siap di-split.")

Preprocessing selesai. Data siap di-split.


### **Train-Test Splitting**

#### **Data Test**

In [8]:
test_clean = df[df['SalePrice'].isnull()].copy()

X_test_final = test_clean.drop(columns=['SalePrice', 'Id'])

#### **Data Train**

In [ ]:
from sklearn.model_selection import train_test_split

train_clean = df[df['SalePrice'].notnull()].copy()

X = train_clean.drop(columns=['SalePrice', 'Id'])
y = train_clean['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## **Model Making**

### **Early Setting**

In [ ]:
# Pisahkan deteksi kolom kategorikal dan numerikal
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Imputasi awal agar model dan encoder tidak gagal saat mendeteksi NaN
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')

X_base = X.copy()
X_base[num_cols] = num_imputer.fit_transform(X_base[num_cols])
X_base[cat_cols] = cat_imputer.fit_transform(X_base[cat_cols])

# Global dictionary penampung hasil
oof_scores = {}
best_params = {}

### **Setting Hyperparameter Tuning**

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Setting 10-Fold Cross Validation secara Global
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Buat tempat penampungan prediksi OOF (data train) dan prediksi Data Test Final
oof_lgb = np.zeros(int(len(X)))
oof_xgb = np.zeros(int(len(X)))
oof_cat = np.zeros(int(len(X)))
oof_en = np.zeros(int(len(X)))
oof_mlp = np.zeros(int(len(X)))

pred_test_en = np.zeros(int(len(X_test_final)))
pred_test_mlp = np.zeros(int(len(X_test_final)))
pred_test_lgb = np.zeros(int(len(X_test_final)))
pred_test_xgb = np.zeros(int(len(X_test_final)))
pred_test_cat = np.zeros(int(len(X_test_final)))


### **XGBoost**

In [12]:
# ==========================================
# A. TUNING & TRAINING XGBOOST (FIXED)
# ==========================================
def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror', 
        'eval_metric': 'rmse', 
        'enable_categorical': True, 
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_jobs': -1,
        'early_stopping_rounds': 50 # PABRIK BARU: Dipindah ke sini agar tidak error pas .fit()
    }
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        # SEKARANG AMAN: early_stopping_rounds dihapus dari fit()
        model.fit(
            X_tr, y_tr, 
            eval_set=[(X_va, y_va)], 
            verbose=False
        )
        preds = model.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("\nMemulai Tuning XGBoost...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=15)
best_params_xgb = study_xgb.best_params
best_params_xgb.update({
    'objective': 'reg:squarederror', 
    'eval_metric': 'rmse', 
    'enable_categorical': True, 
    'random_state': 42, 
    'n_jobs': -1,
    'early_stopping_rounds': 50 # Masukkan juga ke parameter training final
})

# Training 10-Fold Final untuk XGBoost
for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**best_params_xgb)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = model.predict(X_va)
    pred_test_xgb += model.predict(X_test_final) / kf.n_splits

print(f"-> RMSLE OOF XGBoost: {root_mean_squared_error(y, oof_xgb):.5f}")


Memulai Tuning XGBoost...
-> RMSLE OOF XGBoost: 0.12360


### **LightGBM**

In [13]:
# ==========================================
# B. TUNING & TRAINING LIGHTGBM (FAST MODE)
# ==========================================
def objective_lgb(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'random_state': 42, 'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500), # Diturunkan biar cepat
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_jobs': -1 # Poin 3: Pakai semua core CPU untuk LightGBM
    }
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        # Tambahkan callback early stopping bawaan lightgbm
        model.fit(
            X_tr, y_tr, 
            eval_set=[(X_va, y_va)], 
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        preds = model.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("\nMemulai Tuning LightGBM...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=15)
best_params_lgb = study_lgb.best_params
best_params_lgb.update({'objective': 'regression', 'metric': 'rmse', 'random_state': 42, 'verbose': -1, 'n_jobs': -1})

# Training 10-Fold Final untuk LightGBM
for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**best_params_lgb)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = model.predict(X_va)
    pred_test_lgb += model.predict(X_test_final) / kf.n_splits

print(f"-> RMSLE OOF LightGBM: {root_mean_squared_error(y, oof_lgb):.5f}")


Memulai Tuning LightGBM...
-> RMSLE OOF LightGBM: 0.12595


### **CatBoost**

In [ ]:
from catboost import CatBoostRegressor
# Jika ingin pakai pruning otomatis dari Optuna:
# from optuna.integration import CatBoostPruningCallback

# ==========================================
# C. TUNING & TRAINING CATBOOST (OPTIMIZED FAST MODE)
# ==========================================

# 1. Definisikan cat_features SEKALI saja di luar loop
cat_features = X.select_dtypes(include=['category']).columns.tolist()

def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 7),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_seed': 42,
        'verbose': 0,
        'thread_count': -1
    }
    
    # STRATEGI FAST MODE: Gunakan 1-Fold (Single Split) saja untuk Tuning Optuna
    # Ambil fold pertama dari kf untuk menghemat waktu secara drastis
    train_idx, val_idx = next(kf.split(X, y))
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**params)
    model.fit(
        X_tr, y_tr, 
        eval_set=(X_va, y_va), 
        early_stopping_rounds=50,
        cat_features=cat_features, # Menggunakan variabel luar
        verbose=0
    )
    
    preds = model.predict(X_va)
    return root_mean_squared_error(y_va, preds)

print("Memulai Tuning CatBoost (Fast Mode)...")
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=15)

best_params_cat = study_cat.best_params
best_params_cat.update({'verbose': 0, 'random_seed': 42, 'thread_count': -1})

# 2. Training Final Menggunakan Full K-Fold (Dapatkan OOF & Prediksi Test)
print("Memulai Final K-Fold Training dengan Best Params...")
for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**best_params_cat)
    model.fit(
        X_tr, y_tr, 
        eval_set=(X_va, y_va), 
        early_stopping_rounds=50,
        cat_features=cat_features, 
        verbose=0
    )
    
    oof_cat[val_idx] = model.predict(X_va)
    pred_test_cat += model.predict(X_test_final) / kf.n_splits

print(f"-> RMSLE OOF CatBoost: {root_mean_squared_error(y, oof_cat):.5f}")

Memulai Tuning CatBoost (Fast Mode)...
Memulai Final K-Fold Training dengan Best Params...
-> RMSLE OOF CatBoost: 0.12399


In [107]:
# ==========================================
# D. TUNING & TRAINING ELASTICNET
# ==========================================
from sklearn.linear_model import ElasticNet

def objective_en(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 0.9),
        'max_iter': 5000,
        'random_state': 42
    }
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = ElasticNet(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("Memulai Tuning ElasticNet...")
study_en = optuna.create_study(direction='minimize')
study_en.optimize(objective_en, n_trials=20)
best_params_en = study_en.best_params
best_params_en.update({'max_iter': 5000, 'random_state': 42})

# Training 10-Fold Final untuk ElasticNet
for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = ElasticNet(**best_params_en)
    model.fit(X_tr, y_tr)
    oof_en[val_idx] = model.predict(X_va)
    pred_test_en += model.predict(X_test_final) / kf.n_splits

print(f"-> RMSLE OOF ElasticNet: {root_mean_squared_error(y, oof_en):.5f}")

Memulai Tuning ElasticNet...


[W 2026-07-08 09:38:59,116] Trial 0 failed with parameters: {'alpha': 0.0009720545332326764, 'l1_ratio': 0.22393309477907747} because of the following error: ValueError("could not convert string to float: 'RL'").
Traceback (most recent call last):
  File "c:\Users\danis\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\danis\AppData\Local\Temp\ipykernel_20632\3501242946.py", line 19, in objective_en
    model.fit(X_tr, y_tr)
    ~~~~~~~~~^^^^^^^^^^^^
  File "c:\Users\danis\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\danis\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py", line 1149, in fit
    X, y = validate_data(
           ~~~~~~~~~~~~~^
        self,
        ^^^^^
    ...<9 lines>...
        y_numeric=

ValueError: could not convert string to float: 'RL'

In [ ]:
# ==========================================
# E. TUNING & TRAINING MLP REGRESSOR (Neural Network)
# ==========================================
from sklearn.neural_network import MLPRegressor

def objective_mlp(trial):
    # Setup arsitektur neural network layer tersembunyi
    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(64, 32), (128, 64), (64, 64)])
    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': 'relu',
        'solver': 'adam',
        'alpha': trial.suggest_float('alpha', 1e-4, 1.0, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-3, 1e-1, log=True),
        'max_iter': 1000,
        'random_state': 42,
        'early_stopping': True, # Menggunakan internal early stopping bawaan sklearn
        'n_iter_no_change': 20
    }
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = MLPRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("\nMemulai Tuning MLPRegressor...")
study_mlp = optuna.create_study(direction='minimize')
study_mlp.optimize(objective_mlp, n_trials=15)
best_params_mlp = study_mlp.best_params
best_params_mlp.update({'activation': 'relu', 'solver': 'adam', 'max_iter': 1000, 'random_state': 42, 'early_stopping': True, 'n_iter_no_change': 20})

# Training 10-Fold Final untuk MLPRegressor
for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    model = MLPRegressor(**best_params_mlp)
    model.fit(X_tr, y_tr)
    oof_mlp[val_idx] = model.predict(X_va)
    pred_test_mlp += model.predict(X_test_final) / kf.n_splits

print(f"-> RMSLE OOF MLPRegressor: {root_mean_squared_error(y, oof_mlp):.5f}")

In [ ]:
# =========================================================================
# CELL KHUSUS: REKAPITULASI PARAMETER TERBAIK MODEL LEVEL 1
# =========================================================================

print("=" * 60)
print("       📦 REKAPITULASI PARAMETER TERBAIK MODEL LEVEL 1 📦       ")
print("=" * 60)

print(f"\n[1] PARAMETER TERBAIK XGBOOST:")
print(study_xgb.best_params)
print(f"    ⭐ RMSLE OOF Final: {root_mean_squared_error(y, oof_xgb):.5f}")
print("-" * 50)

print(f"\n[2] PARAMETER TERBAIK LIGHTGBM:")
print(study_lgb.best_params)
print(f"    ⭐ RMSLE OOF Final: {root_mean_squared_error(y, oof_lgb):.5f}")
print("-" * 50)

print(f"\n[3] PARAMETER TERBAIK CATBOOST:")
print(study_cat.best_params)
print(f"    ⭐ RMSLE OOF Final: {root_mean_squared_error(y, oof_cat):.5f}")
print("-" * 50)

print(f"\n[4] PARAMETER TERBAIK ELASTICNET:")
print(study_en.best_params)
print(f"    ⭐ RMSLE OOF Final: {root_mean_squared_error(y, oof_en):.5f}")
print("-" * 50)

print(f"\n[5] PARAMETER TERBAIK MLPREGRESSOR:")
print(study_mlp.best_params)
print(f"    ⭐ RMSLE OOF Final: {root_mean_squared_error(y, oof_mlp):.5f}")
print("=" * 60)

### **Dataframe for Meta-Model**

In [ ]:
# =========================================================================
# CELL 7: MEMBUAT DATAFRAME META-FEATURES BARU (5 ASISTEN)
# =========================================================================

# 1. Satukan 5 Prediksi OOF menjadi Dataframe Train untuk Meta-Model
X_meta_train = pd.DataFrame({
    'Pred_XGBoost': oof_xgb,
    'Pred_LightGBM': oof_lgb,
    'Pred_CatBoost': oof_cat,
    'Pred_ElasticNet': oof_en,
    'Pred_MLPRegressor': oof_mlp
})

# 2. Satukan 5 Prediksi Test Final menjadi Dataframe Test untuk Meta-Model
X_meta_test = pd.DataFrame({
    'Pred_XGBoost': pred_test_xgb,
    'Pred_LightGBM': pred_test_lgb,
    'Pred_CatBoost': pred_test_cat,
    'Pred_ElasticNet': pred_test_en,
    'Pred_MLPRegressor': pred_test_mlp
})

print("DataFrame Meta-Features Master (5 Kolom) Berhasil Dibuat!")
print(f"Dimensi X_meta_train: {X_meta_train.shape}")
print("\nIntip data teratas:")
print(X_meta_train.head())

DataFrame Meta-Features Berhasil Dibuat!

Contoh struktur X_meta_train:
   Pred_LightGBM  Pred_XGBoost  Pred_CatBoost
0      12.233897     12.228081      12.224823
1      12.032680     12.078836      12.061773
2      12.250813     12.243559      12.275197
3      12.191026     12.249250      12.167319
4      12.645438     12.688455      12.667073


### **Meta-Model**

In [ ]:
from sklearn.linear_model import Lasso, Ridge
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

# Matikan log optuna biar rapi
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. TUNING META-MODEL: LASSO REGRESSION
# ==========================================
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)
    scores = []
    for train_idx, val_idx in kf.split(X_meta_train, y):
        X_tr, X_va = X_meta_train.iloc[train_idx], X_meta_train.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        meta_lasso = Lasso(alpha=alpha, max_iter=10000, random_state=42)
        meta_lasso.fit(X_tr, y_tr)
        preds = meta_lasso.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("Tuning Meta-Lasso...")
study_lasso = optuna.create_study(direction='minimize')
study_lasso.optimize(objective_lasso, n_trials=30)
print(f"-> Terbaik Lasso RMSLE: {study_lasso.best_value:.5f} dengan Alpha: {study_lasso.best_params['alpha']}")

Memulai Tuning Meta-Model Ridge Regression...

-> Alpha (Lambda) Terbaik untuk Ridge: 1.81193
==> SKOR LOKAL FINAL RMSLE STACKING: 0.11982


In [ ]:
# ==========================================
# 2. TUNING META-MODEL: RIDGE REGRESSION
# ==========================================
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
    scores = []
    for train_idx, val_idx in kf.split(X_meta_train, y):
        X_tr, X_va = X_meta_train.iloc[train_idx], X_meta_train.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        meta_ridge = Ridge(alpha=alpha, random_state=42)
        meta_ridge.fit(X_tr, y_tr)
        preds = meta_ridge.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("\nTuning Meta-Ridge...")
study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(objective_ridge, n_trials=30)
print(f"-> Terbaik Ridge RMSLE: {study_ridge.best_value:.5f} dengan Alpha: {study_ridge.best_params['alpha']}")

In [ ]:
# ==========================================
# 3. TUNING META-MODEL: LIGHTGBM MINI (Max Depth = 3)
# ==========================================
def objective_lgb_meta(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'random_state': 42, 'verbose': -1,
        'max_depth': 3, # Dikunci mini agar tidak gampang overfitting
        'num_leaves': trial.suggest_int('num_leaves', 3, 7),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
        'n_jobs': -1
    }
    scores = []
    for train_idx, val_idx in kf.split(X_meta_train, y):
        X_tr, X_va = X_meta_train.iloc[train_idx], X_meta_train.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        meta_lgb = lgb.LGBMRegressor(**params)
        meta_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(20, verbose=False)])
        preds = meta_lgb.predict(X_va)
        scores.append(root_mean_squared_error(y_va, preds))
    return np.mean(scores)

print("\nTuning Meta-LightGBM Mini...")
study_lgb_meta = optuna.create_study(direction='minimize')
study_lgb_meta.optimize(objective_lgb_meta, n_trials=20)
# PERBAIKAN: Ditambahkan print output instan setelah code model selesai run agar konsisten
print(f"-> Terbaik Meta-LGBM Mini RMSLE: {study_lgb_meta.best_value:.5f}")

In [ ]:
# =========================================================================
# CELL KHUSUS: REKAPITULASI PARAMETER TERBAIK & RMSLE META-MODEL
# (Taruh di cell terpisah paling bawah)
# =========================================================================

print("=" * 60)
print("          🏆 REKAPITULASI AUDISI FINAL META-MODEL 🏆          ")
print("=" * 60)

# 1. Tampilkan Hasil Lasso
print(f"\n[1] META-MODEL: LASSO REGRESSION")
print(f"    ⭐ RMSLE Skor : {study_lasso.best_value:.5f}")
print(f"    📦 Best Params: {study_lasso.best_params}")
print("-" * 50)

# 2. Tampilkan Hasil Ridge
print(f"\n[2] META-MODEL: RIDGE REGRESSION")
print(f"    ⭐ RMSLE Skor : {study_ridge.best_value:.5f}")
print(f"    📦 Best Params: {study_ridge.best_params}")
print("-" * 50)

# 3. Tampilkan Hasil LightGBM Meta
print(f"\n[3] META-MODEL: LIGHTGBM MINI (MAX DEPTH = 3)")
print(f"    ⭐ RMSLE Skor : {study_lgb_meta.best_value:.5f}")
print(f"    📦 Best Params: {study_lgb_meta.best_params}")

print("\n" + "=" * 60)
print("Silakan pilih model dengan RMSLE terkecil untuk submit ke Kaggle!")
print("=" * 60)

## **Evaluate Model Performance**

### **First Evaluation**

In [ ]:
# import time
# from sklearn.metrics import root_mean_squared_error

# # ==========================================
# # 1. EVALUASI DI DATA VALIDASI (X_val)
# # ==========================================

# # --- LightGBM ---
# start_time = time.time()
# lgb_val_pred = lgb_model.predict(X_val)
# lgb_time = time.time() - start_time
# lgb_rmsle = root_mean_squared_error(y_val, lgb_val_pred)

# # --- XGBoost ---
# start_time = time.time()
# xgb_val_pred = xgb_model.predict(X_val)
# xgb_time = time.time() - start_time
# xgb_rmsle = root_mean_squared_error(y_val, xgb_val_pred)


# # ==========================================
# # 2. BUAT TABEL KOMPARASI KINERJA
# # ==========================================
# df_komparasi = pd.DataFrame({
#     'Model': ['LightGBM', 'XGBoost'],
#     'RMSLE (Validation)': [lgb_rmsle, xgb_rmsle],
#     'Inference Time (Seconds)': [lgb_time, xgb_time]
# })

# print("--- TABEL KOMPARASI BASELINE MODEL ---")
# print(df_komparasi.to_string(index=False))
# print("-" * 38)

## **Update and Ensemble**

### **First Attempt**

**Penjelasan:**  
- Data train-test dari train.csv dibagi dengan ratio 80:20 
- Menggunakan data mentah tanpa adanya Feature Engineering
- End pertama ini menggunakan 2 model, yaitu LightGBM dan XGBoost 
- Kedua model di-ensemble dan hasilnya lebih akurat di metrik RMSLE 

RMSLE LightGBM : 0.13397  
RMSLE XGBoost  : 0.14012  
RMSLE Blended  : 0.13190  

### **Second Attempt**

**Penjelasan:**  
- menggunakan LGBM, XGB, CatBoost, dan meta-model (Ridge Regression)
- Menggunakan data mentah tanpa adanya Feature Engineering
- Menggunakan Hyperparameter tuning dengan optuna dan 10 fold CV

RMSLE OOF XGBoost: 0.12415  
RMSLE OOF LightGBM: 0.12627  
RMSLE OOF CatBoost: 0.11914  
SKOR LOKAL FINAL RMSLE STACKING: 0.11734  

In [ ]:
# lgb_final_pred = np.expm1(lgb_model.predict(X_test_final))
# xgb_final_pred = np.expm1(xgb_model.predict(X_test_final))

In [ ]:
# # 1. Gabungkan prediksi data validasi (Bobot: 70% LightGBM, 30% XGBoost)
# blended_val_pred = (0.7 * lgb_val_pred) + (0.3 * xgb_val_pred)

# # 2. Hitung nilai RMSLE hasil gabungan
# blended_rmsle = root_mean_squared_error(y_val, blended_val_pred)

# # 3. Cetak hasil perbandingan
# print(f"RMSLE LightGBM : {lgb_rmsle:.5f}")
# print(f"RMSLE XGBoost  : {xgb_rmsle:.5f}")
# print(f"RMSLE Blended  : {blended_rmsle:.5f}")

## **Submission Kaggle**

In [ ]:
# # Buat draf dataframe submission untuk Kaggle (Ganti model_pred dengan yang skornya paling kecil)
# # Misalnya kita pakai LightGBM dulu:
# submission = pd.DataFrame({
#     'Id': test_clean['Id'],
#     'SalePrice': final_ensemble_pred
# })

# # Simpan ke CSV jika lu mau cek hasilnya
# submission.to_csv('submission.csv', index=False)
# print("Prediksi untuk data X_test_final selesai dibuat!")